# Assignment 1 — Power Constraint Matching Pipeline

**Author:** Yingyun Zhan

The goal of this assignment is to match PJMISO constraints across three data sources: Market, Dayzer, and Panorama/Pano.

I treated this as a record-matching problem. Each source describes a physical constraint in a slightly different way, but the key idea is still the same: a constraint is mainly defined by a monitored facility and a contingency. Therefore, my workflow is:

1. load the three raw files;
2. convert them into a common format;
3. clean and normalize the text fields;
4. use fuzzy matching to find the closest Dayzer and Pano records for each Market constraint;
5. export a result table with scores and review flags.

Most of the reusable code is placed in `pipeline.py`, and this notebook shows the main steps and checks the output.

## 1. Setup

I use `pandas` for data handling and a helper module called `pipeline.py` for the matching logic. The script uses `rapidfuzz` if it is installed because it is faster for fuzzy string matching. I also kept a backup scoring function in the script so the code can still run even if `rapidfuzz` is not available.

In [29]:
import pandas as pd
import pipeline as cp   # our single source of truth

pd.set_option('display.max_colwidth', 60)
print('Scoring backend:', cp.SCORER_BACKEND)

Scoring backend: python-fallback


## 2. Load the raw files

The three files use different column names. Market and Pano already separate the monitored facility and contingency, while Dayzer stores both pieces inside the `NAME` column. I first inspect the shapes and first few rows to understand the structure of each source.

In [30]:
market = pd.read_csv('Market_PJMISO_constraint_list.csv')
dayzer = pd.read_csv('Dayzer_PJMISO_constraint_list.csv')
pano   = pd.read_csv('Pano_PJMISO_constraint_list.csv')

print('Market', market.shape, '| Dayzer', dayzer.shape, '| Pano', pano.shape)
market.head(3)

Market (5230, 6) | Dayzer (13813, 2) | Pano (21963, 5)


,CONSTRAINT,CONTINGENCY,TOZONE,REPORTEDNAME,CONSTRAINTID,CONTINGENCYID
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV,L500.CONASTONE-PEACHBOTTOM.5012,NaN,NOTTINGHM 2-3 SER DEV A 230 KV,10002384305,10002493264
1,LENOX 115 KV LENOX-NMESHOPP NML 1090,L230.ETOWANDA-HILLSIDE.2002 [NYISO],PENELEC,LENOX-NMESHOPP NML 1090 B 115 KV,10000566138,10001865201
2,EASTON 69 KV EAS-EMU,ACTUAL,DPL,EASTON 69 KV EAS-EMU,10004072634,10000680485


In [31]:
dayzer.head(3)

,CID,NAME
0,1,Eastern Interface
1,2,Central Interface
2,3,Western Interface


In [32]:
pano.head(3)

,PID,Monitored Facility,Contingency Name,Earliest,Latest
0,1,CHIC_AVE 138KV - PRAXAIR3 138KV (CHIC_AVE 138 KV CHI-PRA2),L765.Dumont-WiltonCenter.11215,2017-06-29 00:00:00-04:00,2026-04-28 23:00:00-04:00
1,2,177 BURN 345KV - MUNSTER2 345KV (177 BURN 345 KV BUR-MUN1),L765.Dumont-WiltonCenter.11215,2016-11-23 03:00:00-05:00,2026-04-28 23:00:00-04:00
2,3,CONASTON 500KV - CONASTON 230KV (CONASTON 500 KV 500-4),L500.Brighton-Conastone.5011,2021-02-17 06:00:00-05:00,2026-04-28 23:00:00-04:00


## 3. Transform — Standardize the schemas

Before matching, I need the three sources to have comparable fields. I convert each table into the same basic structure:

`source`, `id`, `raw_name`, `facility_raw`, `contingency_raw`

For Market and Pano, the facility and contingency are already separate columns. For Dayzer, I split the `NAME` field at the first colon. The part before the colon is treated as the facility, and the part after the colon is treated as the contingency. If there is no colon, I keep the row and leave the contingency blank.

In [33]:
market_std = cp.standardize_market(market)
dayzer_std = cp.standardize_dayzer(dayzer)
pano_std   = cp.standardize_pano(pano)   # keeps ALL rows; carries the Latest date

market_std.head(3)

,source,id,facility_raw,contingency_raw,raw_name
0,market,10002384305_10002493264,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV,L500.CONASTONE-PEACHBOTTOM.5012,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.CONASTONE-P...
1,market,10000566138_10001865201,LENOX 115 KV LENOX-NMESHOPP NML 1090,L230.ETOWANDA-HILLSIDE.2002 [NYISO],LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ETOWANDA-HIL...
2,market,10004072634_10000680485,EASTON 69 KV EAS-EMU,ACTUAL,EASTON 69 KV EAS-EMU : ACTUAL


## 4. Transform — Clean and normalize text

The raw names contain many formatting differences, such as capitalization, punctuation, voltage notation, and generic words. For example, `138KV`, `138 KV`, and `138-kv` should be treated as the same voltage level.

The cleaning step:
- converts text to uppercase;
- standardizes voltage notation;
- removes punctuation and extra spaces;
- removes generic words such as `LINE`, `TRANSFORMER`, and `MONITORED`;
- converts blank / `ACTUAL` / `BASE` contingencies into a common `BASE` label.

I did this because fuzzy matching works better when the strings are cleaned into a more consistent format.

In [34]:
market_std = cp.add_clean_fields(market_std)
dayzer_std = cp.add_clean_fields(dayzer_std)
pano_std   = cp.add_clean_fields(pano_std)

market_std[['facility_clean', 'contingency_clean', 'voltage']].head(5)

,facility_clean,contingency_clean,voltage
0,NOTTINGH 230 KV NOTTINGHM 2 3 SER DEV,L500 CONASTONE PEACHBOTTOM 5012,230
1,LENOX 115 KV LENOX NMESHOPP NML 1090,L230 ETOWANDA HILLSIDE 2002 NYISO,115
2,EASTON 69 KV EAS EMU,BASE,69
3,SAYRECON 230 KV SAY SAY,BASE,230
4,MOUN UGI 230 KV MOUN UGI 2,230 66 MOUNTAIN T1 SCTNLZ,230


In [35]:
# Normalisation sharply reduces UNKNOWN voltage vs. parsing the raw field
for name, df in [('Market', market_std), ('Dayzer', dayzer_std), ('Pano', pano_std)]:
    unk = (df['voltage'] == 'UNKNOWN').mean() * 100
    print(f'{name:7} UNKNOWN voltage: {unk:5.1f}%  ({len(df):,} rows)')

Market  UNKNOWN voltage:   4.7%  (5,230 rows)
Dayzer  UNKNOWN voltage:   2.0%  (13,813 rows)
Pano    UNKNOWN voltage:   3.7%  (21,963 rows)


## 5. Matching method

For each Market constraint, I search for the closest Dayzer record and the closest Pano record.

I compare the facility part and the contingency part separately. Then I combine the two scores:

`final_score = 0.65 * facility_score + 0.35 * contingency_score`

I give more weight to the facility because the monitored facility is usually the most stable part of the constraint name. Contingency names can be written in more inconsistent ways across sources.

To make the matching faster, I do not compare every Market row with every possible candidate. Instead, I first use shared facility tokens to create a smaller candidate list. For example, if a Market constraint contains a station name, I only compare it with candidates that share at least one meaningful token. This keeps the logic efficient while still allowing flexible fuzzy matching.

Voltage is used as supporting information, but I do not use it as a strict filter. This is because some transformer constraints may contain more than one voltage level.

In [36]:
dayzer_matches = cp.run_matching(market_std, dayzer_std, 'Dayzer')
pano_matches   = cp.run_matching(market_std, pano_std,   'Pano')
dayzer_matches.head(3)

Matching Market -> Dayzer ...
  1,000/5,230 rows
  2,000/5,230 rows
  3,000/5,230 rows
  4,000/5,230 rows
  5,000/5,230 rows
Matching Market -> Pano ...
  1,000/5,230 rows
  2,000/5,230 rows
  3,000/5,230 rows
  4,000/5,230 rows
  5,000/5,230 rows


,matched_id,matched_raw_name,facility_score,contingency_score,final_score
0,532145,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom.5012,100.0,100.0,100.00
1,529457,LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.2002 [NYISO]...,75.0,100.0,83.75
2,520100,EASTON_69 KV_EAS-EMU:ACTUAL,100.0,100.0,100.00


## 6. Build the result table

Required columns first (`market_constraint`, `dayzer_constraint`, `pano_constraint`), then per-source score / confidence / status, the Pano `Latest` date with a `pano_stale` flag, and an `overall` confidence taken as the **weaker** of the two matches so we never over-claim a three-way link.

Confidence bands: `high ≥ 90`, `medium ≥ 80`, `low ≥ 70`, else `review`.

In [37]:
result = cp.build_result_table(market_std, dayzer_matches, pano_matches, pano_std)
result[['market_constraint', 'dayzer_constraint', 'pano_constraint',
        'dayzer_score', 'pano_score', 'overall_confidence']].head(5)

,market_constraint,dayzer_constraint,pano_constraint,dayzer_score,pano_score,overall_confidence
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.CONASTONE-P...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom.5012,NOTTINGH230 KV 2-3 : (NUKE) L500.PeachBottom-Conastone....,100.00,100.00,high
1,LENOX 115 KV LENOX-NMESHOPP NML 1090 : L230.ETOWANDA-HIL...,LENOX_115 KV_LEN-NME:L230.ETowanda-Hillside.2002 [NYISO]...,NMESHOPP 115KV - LENOX 115KV (LENOX 115 KV LEN-NME) : L2...,83.75,90.08,medium
2,EASTON 69 KV EAS-EMU : ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU) : BASE,100.00,100.00,high
3,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 KV SAY-SAY...,100.00,100.00,high
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MOUNTAIN.T1 ...,MOUN UGI_230 KV_2:230/66.Mountain.T1 (Sctnlz),MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 KV 2) : 230...,100.00,100.00,high


## 7. Quality analysis

In [38]:
print('Dayzer confidence:')
print(result['dayzer_confidence'].value_counts())
print('\nPano confidence:')
print(result['pano_confidence'].value_counts())
print('\nOverall confidence:')
print(result['overall_confidence'].value_counts())

Dayzer confidence:
dayzer_confidence
high      3493
medium    1276
low        238
review     223
Name: count, dtype: int64

Pano confidence:
pano_confidence
high      4581
medium     483
low        126
review      40
Name: count, dtype: int64

Overall confidence:
overall_confidence
high      3380
medium    1335
low        279
review     236
Name: count, dtype: int64


In [39]:
for col in ['dayzer_score', 'pano_score']:
    s = result[col]
    print(f'{col}:  >=70 {(s>=70).mean()*100:4.1f}%   >=90 {(s>=90).mean()*100:4.1f}%')

stale = result.loc[result['pano_status']=='matched', 'pano_stale'].mean()*100
print(f'\nAmong matched Pano rows, {stale:.0f}% point to a historical (stale) constraint.')

dayzer_score:  >=70 95.7%   >=90 66.8%
pano_score:  >=70 99.2%   >=90 87.6%

Among matched Pano rows, 87% point to a historical (stale) constraint.


## 8. Spot checks

Confident three-way matches, and weak ones the pipeline correctly flags for review.

In [40]:
hi = result[result['overall_confidence']=='high']
hi[['market_constraint','dayzer_constraint','pano_constraint']].head(4)

,market_constraint,dayzer_constraint,pano_constraint
0,NOTTINGH 230 KV NOTTINGHM 2-3 SER DEV : L500.CONASTONE-P...,NOTTINGH_230 KV_2-3:L500.Conastone-PeachBottom.5012,NOTTINGH230 KV 2-3 : (NUKE) L500.PeachBottom-Conastone....
2,EASTON 69 KV EAS-EMU : ACTUAL,EASTON_69 KV_EAS-EMU:ACTUAL,EASTON 69KV - EMUNI 69KV (EASTON 69 KV EAS-EMU) : BASE
3,SAYRECON230 KV SAY-SAY : ACTUAL,SAYRECON_230 KV_SAY-SAY:ACTUAL,SAYREVIL 230KV - SAYRECON 230KV (SAYRECON 230 KV SAY-SAY...
4,MOUN UGI 230 KV MOUN UGI 2 XFORMER : 230/66.MOUNTAIN.T1 ...,MOUN UGI_230 KV_2:230/66.Mountain.T1 (Sctnlz),MOUN UGI 69KV - MOUN UGI 230KV (MOUN UGI 230 KV 2) : 230...


In [41]:
rev = result[result['overall_confidence']=='review']
rev[['market_constraint','dayzer_score','dayzer_status','pano_score','pano_status']].head(5)

,market_constraint,dayzer_score,dayzer_status,pano_score,pano_status
15,CHICAGO-PRAXAIR3 138 KV L/O WILTON CENTER-DUMONT 765 KV ...,51.98,review,87.30,matched
32,PREST - TIBBS 138 KV L/O ASTER - COMMODORE 345 KV : ASTE...,74.84,matched,59.61,review
46,STILLWELL-DUMONT 345 L/O WILTON CENTER-DUMONT 765 : L765...,59.89,review,91.22,matched
48,PA-CENT CONTINGENCY 1 : L500.JUNIATA-SUNBURY.5046,52.72,review,91.88,matched
50,SUB85_SUB18_161_FLO_OAKGROVE_LOUISA_345 : L345.OAKGROVE-...,48.06,unmatched,57.14,review


## 9. Load — export


In [42]:
result.to_csv('constraint_mapping_results.csv', index=False)
print('Exported', result.shape, '-> constraint_mapping_results.csv')

Exported (5230, 16) -> constraint_mapping_results.csv
